# MegaDetector Setup (third-party/eb_megadetector)

This notebook provides a step-by-step setup path for the local MegaDetector dependency under:

- `third-party/eb_megadetector`

It is intended for initial bootstrap and repeatable environment validation.

## Step 1 — Confirm repository root and MegaDetector path

In [ ]:
from pathlib import Path
import subprocess

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
md_path = repo_root / "third-party" / "eb_megadetector"

print(f"Repository root: {repo_root}")
print(f"MegaDetector path: {md_path}")
print(f"Exists: {md_path.exists()}")

## Step 2 — Clone Earth-Biometrics MegaDetector fork if missing

Run the next cell only if `third-party/eb_megadetector` does not exist.

Source repo:
- `git@github.com:Earth-Biometrics/eb_megadetector.git`

In [ ]:
from pathlib import Path
import subprocess

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
third_party_dir = repo_root / "third-party"
md_dir = third_party_dir / "eb_megadetector"
repo_url = "git@github.com:Earth-Biometrics/eb_megadetector.git"

third_party_dir.mkdir(parents=True, exist_ok=True)
if md_dir.exists():
    print(f"MegaDetector already present at {md_dir}")
else:
    subprocess.run(["git", "clone", repo_url, str(md_dir)], check=True)
    print(f"Cloned {repo_url} to {md_dir}")

## Step 3 — Install MegaDetector dependencies

This cell installs from common requirement files if they are present.

In [ ]:
from pathlib import Path
import platform
import subprocess
import sys
import tempfile

repo_root = Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
md_root = repo_root / "third-party" / "eb_megadetector"
candidate_files = [
    md_root / "requirements.txt",
    md_root / "requirements-dev.txt",
    md_root / "envs" / "requirements.txt",
]

existing = [p for p in candidate_files if p.exists()]
print("Requirements files found:")
for req in existing:
    print(f" - {req}")

if not existing:
    raise FileNotFoundError("No requirements file found in expected MegaDetector locations.")

is_linux_arm = sys.platform.startswith("linux") and platform.machine().lower() in {"aarch64", "arm64"}
temp_files = []
try:
    for req in existing:
        install_req = req
        if is_linux_arm:
            lines = req.read_text(encoding="utf-8").splitlines()
            filtered = []
            for line in lines:
                normalized = line.strip().lower().replace(" ", "")
                if normalized.startswith("mkl==") or normalized.startswith("mkl==="):
                    continue
                filtered.append(line)
            if len(filtered) != len(lines):
                tmp = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8")
                with tmp:
                    tmp.write("\n".join(filtered) + "\n")
                install_req = Path(tmp.name)
                temp_files.append(install_req)
                print(f"Linux ARM detected; removed mkl==2024.0 from {req.name} for this install run")

        cmd = [sys.executable, "-m", "pip", "install", "-r", str(install_req)]
        print("Running:", " ".join(cmd))
        subprocess.run(cmd, check=True)

    if is_linux_arm:
        fix_cmd = [sys.executable, "-m", "pip", "install", "clipboard", "ruff", "setuptools<82"]
        print("Running:", " ".join(fix_cmd))
        subprocess.run(fix_cmd, check=True)
        print("Note: MegaDetector's mkl==2024.0 requirement is unavailable on Linux ARM and is intentionally skipped.")
finally:
    for temp_file in temp_files:
        temp_file.unlink(missing_ok=True)

## Step 4 — Smoke test imports

This verifies that core Python dependencies are available.

In [ ]:
import importlib

modules = ["torch", "cv2", "PIL", "numpy"]
results = {}

for name in modules:
    try:
        importlib.import_module(name)
        results[name] = "OK"
    except Exception as exc:
        results[name] = f"FAILED: {exc}"

results

## Step 5 — Next action

After successful setup, run:

- [02_megadetector_local_images.ipynb](/Users/elhorte/git/earth-biometrics/project-id/notebooks/02_megadetector_local_images.ipynb)

to exercise MegaDetector on local images.